In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/raw_reviews.csv")

print(df.shape)
df.head()

(1500, 5)


,review,rating,date,bank,source
0,it's a good application,5,2026-05-13 20:28:58,CBE,Google Play
1,thank you cbe,5,2026-05-13 17:16:37,CBE,Google Play
2,is good,5,2026-05-13 16:18:45,CBE,Google Play
3,wow,5,2026-05-13 12:19:17,CBE,Google Play
4,Good application,2,2026-05-13 11:27:52,CBE,Google Play


In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Lenovo
[nltk_data]     T480s\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import re
import string
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = text.strip()

    # Remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

df["clean_review"] = df["review"].apply(clean_text)

df[["review", "clean_review"]].head()

,review,clean_review
0,it's a good application,good application
1,thank you cbe,thank cbe
2,is good,good
3,wow,wow
4,Good application,good application


In [5]:
from transformers import pipeline

c:\Users\Lenovo T480s\Documents\fintech-review-analytics\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

c:\Users\Lenovo T480s\Documents\fintech-review-analytics\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo T480s\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:0

In [7]:
sentiment_pipeline("This banking app is amazing")

[{'label': 'POSITIVE', 'score': 0.9998641014099121}]

In [8]:
sentiments = []

for review in df["clean_review"]:

    try:
        result = sentiment_pipeline(review[:512])[0]

        sentiments.append({
            "sentiment_label": result["label"],
            "sentiment_score": result["score"]
        })

    except:
        sentiments.append({
            "sentiment_label": "NEUTRAL",
            "sentiment_score": 0
        })

In [ ]:
sentiment_df = pd.DataFrame(sentiments)

df = pd.concat([df, sentiment_df], axis=1)

df.head()

,review,rating,date,bank,source,clean_review,sentiment_label,sentiment_score
0,it's a good application,5,2026-05-13 20:28:58,CBE,Google Play,good application,POSITIVE,0.999855
1,thank you cbe,5,2026-05-13 17:16:37,CBE,Google Play,thank cbe,POSITIVE,0.999794
2,is good,5,2026-05-13 16:18:45,CBE,Google Play,good,POSITIVE,0.999816
3,wow,5,2026-05-13 12:19:17,CBE,Google Play,wow,POSITIVE,0.999592
4,Good application,2,2026-05-13 11:27:52,CBE,Google Play,good application,POSITIVE,0.999855


In [10]:
df.to_csv("../data/processed/sentiment_reviews.csv", index=False)

In [12]:
df.groupby("bank")["sentiment_score"].mean()

bank
BOA       0.960916
CBE       0.967355
Dashen    0.969816
Name: sentiment_score, dtype: float64

In [13]:
df.groupby("rating")["sentiment_score"].mean()

rating
1    0.971274
2    0.955600
3    0.959584
4    0.945713
5    0.967714
Name: sentiment_score, dtype: float64

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
vectorizer = TfidfVectorizer(
    max_features=20,
    ngram_range=(1,2)
)

for bank in df["bank"].unique():

    bank_reviews = df[df["bank"] == bank]["clean_review"]

    X = vectorizer.fit_transform(bank_reviews)

    keywords = vectorizer.get_feature_names_out()

    print(f"\nTop keywords for {bank}:")
    print(keywords)


Top keywords for CBE:
['app' 'application' 'bank' 'best' 'cbe' 'easy' 'excellent' 'fast' 'good'
 'like' 'mobile' 'nice' 'ok' 'please' 'service' 'transfer' 'update' 'use'
 'work' 'working']

Top keywords for BOA:
['app' 'bank' 'banking' 'best' 'boa' 'doesnt' 'even' 'ever' 'fix' 'good'
 'mobile' 'mobile banking' 'nice' 'please' 'time' 'update' 'use' 'work'
 'working' 'worst']

Top keywords for Dashen:
['app' 'bank' 'banking' 'best' 'cant' 'dashen' 'dashen bank' 'easy' 'even'
 'ever' 'fast' 'good' 'great' 'nice' 'one' 'super' 'super app' 'time'
 'use' 'working']


In [19]:
def identify_theme(text):

    text = text.lower()

    if any(word in text for word in ["login", "otp", "password"]):
        return "Account Access Issues"

    elif any(word in text for word in ["transfer", "transaction", "slow"]):
        return "Transaction Performance"

    elif any(word in text for word in ["ui", "interface", "design"]):
        return "UI & UX"

    elif any(word in text for word in ["support", "service"]):
        return "Customer Support"

    else:
        return "General Experience"

df["identified_theme"] = df["clean_review"].apply(identify_theme)

df[["clean_review", "identified_theme"]].head()

,clean_review,identified_theme
0,good application,General Experience
1,thank cbe,General Experience
2,good,General Experience
3,wow,General Experience
4,good application,General Experience


In [20]:
df["identified_theme"].value_counts()

identified_theme
General Experience         1305
Transaction Performance      98
Customer Support             42
UI & UX                      33
Account Access Issues        22
Name: count, dtype: int64

In [21]:
df.to_csv("../data/processed/final_reviews.csv", index=False)